In [0]:
dbutils.library.restartPython()

In [0]:
import json
from datetime import date
from pymongo import MongoClient
from common_utils.ingestor import read_cosmosdb_json, write_raw
from common_utils.logging import get_logger

In [0]:
# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("cosmosdb-ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

# ============================================================
# 2. READ CONFIG
# ============================================================
with open(f"{config_path}", "r") as f:
    config = json.load(f)

# ============================================================
# 3. CONFIG SECTIONS
# ============================================================
source = config["source"]
target = config["target"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat() 
logger.info("Fetching todays date %s", run_date)

connection_string = dbutils.secrets.get(scope="cosmosdb-ingestion", key=source["connection_string"])

# ============================================================
# 5. READ FROM COSMOSDB SERVER
# ============================================================

df = read_cosmosdb_json(spark,connection_string,source["database_name"], source["collection_name"], source["data_type"])
logger.info("Data Succesfully Read")

# ============================================================
# 6. TARGET PATH
# ============================================================
target_path = f"{target["base_path"]}/{target["folder"]}/load_date={run_date}"

# ============================================================
# 7. WRITE RAW
# ============================================================
logger.warning("Writing the in the target %s", target_path)
write_raw(df,target_path,target["file_format"],target["mode"])
logger.info("Data Succesfully written")